# v3.6 TSE Advanced Experiment Template

Use the GUI for FPGA initialisation, SPI, filtering, sequence browsing, and sequence visualization. Load the TSE sequence in the GUI Sequence tab first; API transforms below reprogram transient copies during the scan, while the GUI visualization remains the original loaded sequence.

In [ ]:
import importlib
from IPython.display import display

import src_v3_6.app as gui
from src_v3_6 import core_api
gui = importlib.reload(gui)

display(gui.GUI_total)

## Select And Inspect The Virgin Sequence

In the GUI, open **Sequence -> Sequence Files**, load a TSE sequence such as `TSE_V4R2.txt`, then inspect the timeline/pulse view there. This cell reads the currently loaded GUI sequence as a deep copy for notebook-side inspection only.

In [ ]:
virgin_seq = gui.get_current_sequence()
virgin_seq['ExpConfig']

## Per-Scan Software Transform

This example steps a phase-encoding offset. The transform receives a transient sequence copy for each scan, so it never edits the GUI-loaded baseline or the sequence file.

In [ ]:
phase_encode_offsets = [0, 16, 32, 48]

def tse_transform(seq, context):
    pe = phase_encode_offsets[context.scan_idx % len(phase_encode_offsets)]
    core_api.set_section_field(seq, 'Gpe_pos', 'y_sweep_offset', pe)
    core_api.set_section_field(seq, 'Gpe_neg', 'y_sweep_offset', pe)
    return seq


In [ ]:
# Initialise FPGA from the GUI before this cell.
config = gui.make_experiment_config(
    num_averages=1,
    loop_count=len(phase_encode_offsets),
    tr_ms=1000,
)

def on_scan(result):
    print(f'PE step {result.context.scan_idx + 1}/{result.context.total_scans}')

avg_ch0, avg_ch1 = gui.run_api_sequence(
    config=config,
    sequence_transform=tse_transform,
    scan_callback=on_scan,
)
avg_ch0.shape, avg_ch1.shape